In [193]:
# to download this dataset
# visit this link 
# https://www.kaggle.com/datasets/jiashenliu/515k-hotel-reviews-data-in-europe

import pandas as pd
import time

csv_file = "../../data/Hotel_Reviews.csv"
print(f"loading data from csv file {csv_file}", flush=True)
start = time.time()
df = pd.read_csv(csv_file)
end = time.time()
print(f"loaded {len(df)} rows from file, spent {round(end - start, 2)} senconds")

loading data from csv file ../../data/Hotel_Reviews.csv
loaded 515738 rows from file, spent 0.56 senconds


In [194]:
print(f"the shape of dataset: {df.shape}")

nationality_freq = df.value_counts("Reviewer_Nationality")

print(f"The highest frequency reviewer nationaility is {nationality_freq.index[0].strip()} with {nationality_freq.iloc[0]}")

print("The next 10 highest frequency reviewer national are:\n")
print(nationality_freq[1:11].to_string())

the shape of dataset: (515738, 17)
The highest frequency reviewer nationaility is United Kingdom with 245246
The next 10 highest frequency reviewer national are:

Reviewer_Nationality
United States of America     35437
Australia                    21686
Ireland                      14827
United Arab Emirates         10235
Saudi Arabia                  8951
Netherlands                   8772
Switzerland                   8678
Germany                       7941
Canada                        7894
France                        7296


In [195]:
# get the most visited hotel for each national (top 10)
for nationality in nationality_freq[:10].index:

    nation_df = df[(df["Reviewer_Nationality"] == nationality)]
    hotel_df = nation_df.value_counts("Hotel_Name")

    print(f"the most visited hotel for {nationality}: {hotel_df.index[0]} with {hotel_df.iloc[0]}")

the most visited hotel for  United Kingdom : Britannia International Hotel Canary Wharf with 3833
the most visited hotel for  United States of America : Hotel Esther a with 423
the most visited hotel for  Australia : Park Plaza Westminster Bridge London with 167
the most visited hotel for  Ireland : Copthorne Tara Hotel London Kensington with 239
the most visited hotel for  United Arab Emirates : Millennium Hotel London Knightsbridge with 129
the most visited hotel for  Saudi Arabia : The Cumberland A Guoman Hotel with 142
the most visited hotel for  Netherlands : Jaz Amsterdam with 97
the most visited hotel for  Switzerland : Hotel Da Vinci with 97
the most visited hotel for  Germany : Hotel Da Vinci with 86
the most visited hotel for  Canada : St James Court A Taj Hotel London with 61


In [196]:
hotel_freq_df = df.drop(["Hotel_Address", "Additional_Number_of_Scoring", "Review_Date", "Average_Score", "Reviewer_Nationality", "Negative_Review", "Review_Total_Negative_Word_Counts", "Positive_Review", "Review_Total_Positive_Word_Counts", "Total_Number_of_Reviews_Reviewer_Has_Given", "Reviewer_Score", "Tags", "days_since_review", "lat", "lng"], axis = 1)

hotel_freq_df["Total_Reviews_Found"] = hotel_freq_df.groupby("Hotel_Name").transform("count")

hotel_freq_df = hotel_freq_df.drop_duplicates(subset = ["Hotel_Name"])
display(hotel_freq_df)

,Hotel_Name,Total_Number_of_Reviews,Total_Reviews_Found
0,Hotel Arena,1403,405
405,K K Hotel George,1831,566
971,Apex Temple Court Hotel,2619,1037
2008,The Park Grand London Paddington,4380,1770
3778,Monhotel Lounge SPA,171,35
...,...,...,...
511962,Suite Hotel 900 m zur Oper,3461,439
512401,Hotel Amadeus,717,144
512545,The Berkeley,232,100
512645,Holiday Inn London Kensington,5945,2768


In [197]:
def get_difference_review_avg(row):
    return row["Average_Score"] - row["Calc_Average_Score"]

df["Calc_Average_Score"] = round(df.groupby("Hotel_Name").Reviewer_Score.transform("mean"), 1)

# apply process each row with specific function
# then return a new Series
df["Average_Score_Difference"] = df.apply(get_difference_review_avg, axis=1)

review_scores_df = df.drop_duplicates(subset = ["Hotel_Name"])

review_scores_df = review_scores_df.sort_values(by=["Average_Score_Difference"])

display(review_scores_df[["Average_Score_Difference", "Average_Score", "Calc_Average_Score", "Hotel_Name"]])

,Average_Score_Difference,Average_Score,Calc_Average_Score,Hotel_Name
495945,-0.8,7.7,8.5,Best Western Hotel Astoria
111027,-0.7,8.8,9.5,Hotel Stendhal Place Vend me Paris MGallery by...
43688,-0.7,7.5,8.2,Mercure Paris Porte d Orleans
178253,-0.7,7.9,8.6,Renaissance Paris Vendome Hotel
71274,-0.5,8.9,9.4,Drawing Hotel
...,...,...,...,...
201776,0.7,7.5,6.8,Mercure Paris Op ra Faubourg Montmartre
22189,0.8,7.1,6.3,Holiday Inn Paris Montparnasse Pasteur
250308,0.9,8.6,7.7,MARQUIS Faubourg St Honor Relais Ch teaux
68936,0.9,6.8,5.9,Villa Eugenie


In [198]:
# without lambdas (using a mixture of notations to show you can use both)
# it will much faster then lambda-apply method
start = time.time()
no_negative_reviews = sum(df.Negative_Review == "No Negative")
print("Number of No Negative reviews: " + str(no_negative_reviews))

no_positive_reviews = sum(df["Positive_Review"] == "No Positive")
print("Number of No Positive reviews: " + str(no_positive_reviews))

both_no_reviews = sum((df.Negative_Review == "No Negative") & (df.Positive_Review == "No Positive"))
print("Number of both No Negative and No Positive reviews: " + str(both_no_reviews))

end = time.time()
print("Sum took " + str(round(end - start, 2)) + " seconds")

Number of No Negative reviews: 127890
Number of No Positive reviews: 35946
Number of both No Negative and No Positive reviews: 127
Sum took 0.11 seconds


In [199]:

df = df.drop(["lat", "lng"],axis=1)


def replace_address(row):
    if "Netherlands" in row["Hotel_Address"]:
        return "Amsterdam, Netherlands"
    elif "Barcelona" in row["Hotel_Address"]:
        return "Barcelona, Spain"
    elif "United Kingdom" in row["Hotel_Address"]:
        return "London, United Kingdom"
    elif "Milan" in row["Hotel_Address"]:        
        return "Milan, Italy"
    elif "France" in row["Hotel_Address"]:
        return "Paris, France"
    elif "Vienna" in row["Hotel_Address"]:
        return "Vienna, Austria" 

df["Hotel_Address"] = df.apply(replace_address, axis=1)
print(df["Hotel_Address"].value_counts())


df = df.drop(["Additional_Number_of_Scoring"], axis=1)

df.Total_Number_of_Reviews = df.groupby('Hotel_Name').transform('size')
df.Average_Score = round(df.groupby('Hotel_Name').Reviewer_Score.transform('mean'), 1)


Hotel_Address
London, United Kingdom    262301
Barcelona, Spain           60149
Paris, France              59928
Amsterdam, Netherlands     57214
Vienna, Austria            38939
Milan, Italy               37207
Name: count, dtype: int64


In [200]:
df = df.drop(["Review_Total_Negative_Word_Counts", "Review_Total_Positive_Word_Counts", "Review_Date", "days_since_review"], axis=1)

df = df.drop(["Total_Number_of_Reviews_Reviewer_Has_Given"], axis=1)


In [201]:
# process tags column
# remove the opening and closing brackets
df.Tags = df.Tags.str.strip("[']")
# remove all quotes 
df.Tags = df.Tags.str.replace(" ', '", ",", regex = False)

In [202]:
tag_dict = dict()

def process_tags(row):
    global tag_dict
    for tag in row.split(","):
        # skip the useless tags
        if "Stayed" in tag:
            continue
        elif "Room" in tag or "room" in tag:
            continue
        
        tag = tag.strip(" ")
        if tag_dict.get(tag) is None:
            tag_dict[tag] = 1
            continue
        tag_dict[tag] += 1
df.Tags.apply(process_tags)

tags_se = pd.Series(tag_dict)
tags_se = tags_se.sort_values(axis=0, ascending=False)

In [203]:
tags_se.head(30)

Leisure trip                      417778
Submitted from a mobile device    307640
Couple                            252294
Solo traveler                     108545
Business trip                      82939
Group                              65392
Family with young children         61015
Family with older children         26349
Junior Suite                        4559
Travelers with friends              2143
Suite                               1897
Deluxe Double                       1634
With a pet                          1405
Standard                            1231
Studio                               986
Studio King Family                   925
Studio Suite                         728
Standard Double                      722
Studio Double Family                 506
Superior Shard King                  402
Deluxe with Two Double Beds          391
Double or Twin Ground Floor          384
Luxury Suite                         303
Superior                             270
Double Studio   

In [205]:
# Process the Tags into new columns

df["Leisure_trip"] = df.Tags.apply(lambda tag: 1 if "Leisure trip" in tag else 0)
df["Couple"] = df.Tags.apply(lambda tag: 1 if "Couple" in tag else 0)
df["Solo_traveler"] = df.Tags.apply(lambda tag: 1 if "Solo traverler" in tag else 0)
df["Business_trip"] = df.Tags.apply(lambda tag: 1 if "Business trip" in tag else 0)
df["Group"] = df.Tags.apply(lambda tag: 1 if "Group" in tag else 0)
df["Family_with_young_children"] = df.Tags.apply(lambda tag: 1 if "Family with young children" in tag else 0)
df["Family_with_older_children"] = df.Tags.apply(lambda tag: 1 if "Family with older children" in tag else 0)
df["With_a_pet"] = df.Tags.apply(lambda tag: 1 if "With a pet" in tag else 0)

#df = df.drop(["Review_Total_Negative_Word_Counts", "Review_Total_Positive_Word_Counts", "days_since_review", "Total_Number_of_Reviews_Reviewer_Has_Given"], axis = 1)
df.columns

Index(['Hotel_Address', 'Average_Score', 'Hotel_Name', 'Reviewer_Nationality',
       'Negative_Review', 'Total_Number_of_Reviews', 'Positive_Review',
       'Reviewer_Score', 'Tags', 'Calc_Average_Score',
       'Average_Score_Difference', 'Leisure_trip', 'Couple', 'Solo_traveler',
       'Business_trip', 'Group', 'Family_with_young_children',
       'Family_with_older_children', 'With_a_pet'],
      dtype='str')

In [206]:
out_csv_file = "../../data/Hotel_Reviews_Filtered.csv"

print(f"Saving results to {out_csv_file}")
df.to_csv(out_csv_file, index=False)

Saving results to ../../data/Hotel_Reviews_Filtered.csv
